In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Notebook_Spark") \
    .getOrCreate()

In [ ]:
##Top 2 Products per Country
df = spark.read.csv("sales_data.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("sales")

spark.sql("""
SELECT country, product, total_sales
FROM (
    SELECT 
        country,
        product,
        SUM(amount) AS total_sales,
        ROW_NUMBER() OVER (PARTITION BY country ORDER BY SUM(amount) DESC) AS rank
    FROM sales
    GROUP BY country, product
) t
WHERE rank <= 2
""").show()

In [ ]:
###Top 3 Products by Total Sales
df = spark.read.csv("sales_data.csv", header=True, inferSchema=True)

df.createOrReplaceTempView("sales")
spark.sql("""
SELECT product, SUM(amount) AS total_sales
FROM sales
GROUP BY product
ORDER BY total_sales DESC
LIMIT 3
""").show()


In [ ]:
df = spark.read.csv("sales_data.csv", header=True, inferSchema=True)

df.createOrReplaceTempView("sales")
spark.sql("""
SELECT country, amount AS second_highest_sale
FROM (
    SELECT 
        country,
        amount,
        ROW_NUMBER() OVER (PARTITION BY country ORDER BY amount DESC) AS rank
    FROM sales
) t
WHERE rank = 2
""").show()

In [ ]:
df = spark.read.text("app_logs.txt")
df.createOrReplaceTempView("raw_logs")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW logs AS
SELECT
    get(parts, 0) AS date,
    get(parts, 1) AS time,
    get(parts, 2) AS level,
    get(parts, 3) AS user,
    get(parts, 4) AS action,
    get(parts, 5) AS status_or_item
FROM (
    SELECT split(value, ' ') AS parts
    FROM raw_logs
) t
""")

In [ ]:
#Count of Each Log Level
spark.sql("""
SELECT level, COUNT(*) AS count
FROM logs
GROUP BY level
""").show()

In [ ]:
#Most Active User
spark.sql("""
SELECT user, COUNT(*) AS total_actions
FROM logs
GROUP BY user
ORDER BY total_actions DESC
LIMIT 1
""").show()

In [ ]:
#User with Maximum Errors
spark.sql("""
SELECT user, COUNT(*) AS error_count
FROM logs
WHERE level = 'ERROR'
GROUP BY user
ORDER BY error_count DESC
LIMIT 1
""").show()

In [ ]:
# users with Failed Login Attempts
spark.sql("""
SELECT COUNT(*) AS failed_logins
FROM logs
WHERE action = 'login'
AND status_or_item = 'failed'
""").show()

In [ ]:
#Most Purchased Item
spark.sql("""
SELECT status_or_item AS item,
       COUNT(*) AS purchase_count
FROM logs
WHERE action = 'purchase'
GROUP BY status_or_item
ORDER BY purchase_count DESC
LIMIT 1
""").show()

In [ ]:
#Users with Both Login Success & Failure
spark.sql("""
SELECT DISTINCT l1.user
FROM logs l1
JOIN logs l2
ON l1.user = l2.user

WHERE l1.action = 'login'
AND l1.status_or_item = 'success'

AND l2.action = 'login'
AND l2.status_or_item = 'failed'
""").show()


In [ ]:
#Consecutive Failures
spark.sql("""
SELECT DISTINCT user

FROM (

    SELECT
        user,
        action,
        status_or_item,

        LAG(status_or_item)
        OVER (
            PARTITION BY user
            ORDER BY time
        ) AS prev_status

    FROM logs

) t

WHERE status_or_item = 'failed'
AND prev_status = 'failed'
""").show()